# STAT 3005 — Project 1: Social media use and academic performance
### Data → correlation → full model → multicollinearity (VIF) → variable selection → assumption checks → influential points

Structure follows the Lecture 3 workflow. Every number in the article must be reproducible from this notebook (`data/raw/survey.csv` must be present in the repository).

> **TODO (team):** replace this block with 3–5 sentences on *why* you chose this topic and this survey (the assignment grades the motivation explicitly).
>
> **TODO (team):** state which AI tool was used and for what (e.g. "Claude — help with code syntax and debugging; analysis and conclusions are ours").

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan, linear_reset
from statsmodels.stats.stattools import durbin_watson

plt.rcParams["figure.figsize"] = (7, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
pd.set_option("display.precision", 3)
pd.set_option("display.width", 200)

---
## 1. Data and cleaning
The survey was filled in as free text, so most numeric columns needed parsing (ages like `"21 years old"`, study time like `"3/4h"`, GPA on 4.0 and 5.0 scales and as letters). The cleaning code below is the team's own and is kept unchanged.

In [ ]:
df = pd.read_csv("../data/raw/survey.csv")
df.head()

In [ ]:
df.drop(columns = "Column 19", inplace = True)
df.head()

In [ ]:
print("shape:", df.shape)
print("\n", df.isna().sum())
print(df.drop_duplicates().shape)
print("\n",df.dtypes)

In [ ]:
print(df[df.columns[0]].unique()[:5])

df["Timestamp"] = pd.to_datetime(df["Timestamp"], format="%d/%m/%Y %H:%M:%S")
df["Year"] = df["Timestamp"].dt.year
df["Month"] = df["Timestamp"].dt.month
df["Day"] = df["Timestamp"].dt.day
df["Time"] = df["Timestamp"].dt.time

df.head()

In [ ]:
def clean_age(val):
    if pd.isna(val):
        return np.nan
    val_str = str(val).strip().lower()

    if '210' in val_str:
        return np.nan

    if '/' in val_str:
        try:
            parts = val_str.split('/')
            year = int(parts[-1])
            if 1980 <= year <= 2010:
                return 2025 - year
        except:
            pass

    nums = re.findall(r'\d+', val_str)
    if nums:
        n = int(nums[0])
        if 1980 <= n <= 2010:
            return 2025 - n
        if 10 <= n <= 99:
            return float(n)        
    return np.nan

df['Age'] = df['Age'].apply(clean_age)
df['Age'] = df['Age'].fillna(df['Age'].median()).astype(int)

df.head()

In [ ]:
def parse_study_time(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip().lower()
    if any(k in s for k in ['random', 'varies', 'no specific', 'not fixed', 'depends', 'bsc honours', 'i don\'t have', 'not everyday']):
        return np.nan
    s = s.replace('hrs', 'h').replace('hour', 'h').replace('ours', 'h').replace('approx', '').replace('daily', '')
    s = s.replace('less than', '<').replace('almost', '').replace('at least', '').replace('more than', '>')
    if 'min' in s or 'mins' in s or 'minute' in s:
        nums = re.findall(r'\d+', s)
        return float(nums[0]) / 60.0 if nums else np.nan
    nums = re.findall(r'\d+\.?\d*', s)
    if not nums:
        return np.nan
    float_nums = [float(x) for x in nums]
    if len(float_nums) == 1:
        return float_nums[0]
    elif len(float_nums) >= 2:
        return sum(float_nums[:2]) / 2.0
    return np.nan

df['Study time (In Hours)'] = df['Study time (In Hours)'].apply(parse_study_time)
df['Study time (In Hours)'] = df['Study time (In Hours)'].fillna(df['Study time (In Hours)'].median()).astype(float)

df.head()

In [ ]:
def parse_attendance(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip().lower().replace('%', '')
    nums = re.findall(r'\d+\.?\d*', s)
    if not nums:
        return np.nan
    val_float = float(nums[0])
    if val_float > 1.0:
        val_float = val_float / 100.0
    return val_float if 0.0 <= val_float <= 1.0 else np.nan

df['Attendance rate (In Percentile)'] = df['Attendance rate (In Percentile)'].apply(parse_attendance)
df['Attendance rate (In Percentile)'] = df['Attendance rate (In Percentile)'].fillna(df['Attendance rate (In Percentile)'].median()).astype(float)

df['Attendance rate (In Percentile)'].head()

In [ ]:
def parse_social_time(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip().lower()
    if any(k in s for k in ['sometimes', 'nowadays', 'more than']):
        return np.nan
    s = s.replace('hrs', 'h').replace('hour', 'h').replace('ours', 'h').replace('approx', '').replace('day', '')
    s = s.replace('less than', '<').replace('almost', '').replace('at least', '').replace('around', '').replace('+', '')
    if '%' in s:
        nums = re.findall(r'\d+', s)
        return float(nums[0]) / 10.0 if nums else np.nan
    if 'min' in s or 'mins' in s or 'minute' in s:
        nums = re.findall(r'\d+', s)
        return float(nums[0]) / 60.0 if nums else np.nan
    nums = re.findall(r'\d+\.?\d*', s)
    if not nums:
        return np.nan
    float_nums = [float(x) for x in nums]
    if len(float_nums) == 1:
        return float_nums[0]
    elif len(float_nums) >= 2:
        return sum(float_nums[:2]) / 2.0
    return np.nan

df['Time spent in social media (hours)'] = df['Time spent in social media (hours)'].apply(parse_social_time)
df['Time spent in social media (hours)'] = df['Time spent in social media (hours)'].fillna(df['Time spent in social media (hours)'].median()).astype(float)

df['Time spent in social media (hours)'].head()

In [ ]:
print(df["Withdrawal symptoms (Side effects of not using social media)"].unique())
df["Withdrawal symptoms (Side effects of not using social media)"] = df["Withdrawal symptoms (Side effects of not using social media)"].replace("No symtoms", "No symptoms")

In [ ]:
def standardize_gpa(val):
    if pd.isna(val):
        return None
    val_str = str(val).strip().upper()
    letter_mapping = {
        'A': 4.0, 'A-': 3.67,
        'B+': 3.33, 'B': 3.0, 'B-': 2.67,
        'C+': 2.33, 'C': 2.0, 'C-': 1.67,
        'D+': 1.33, 'D': 1.0,
        'FX': 0.5, 'F': 0.0
    }
    
    if val_str in letter_mapping:
        return letter_mapping[val_str]
    try:
        num = float(val_str)
        if 4.0 < num <= 5.0:
            if num >= 4.5: return 4.0
            elif num >= 3.5: return 3.0
            elif num >= 2.5: return 2.0
            else: return 0.0
        elif 0.0 <= num <= 4.0:
            return float(num)
    except ValueError:
        pass
        
    return None

df["Last Academic Result (GPA/CGPA)"] = df["Last Academic Result (GPA/CGPA)"].apply(standardize_gpa).astype(float)

df["Last Academic Result (GPA/CGPA)"].head()

In [ ]:
print(df["Last Academic Result (GPA/CGPA)"].isna().sum())

In [ ]:
df = df.dropna(subset=['Last Academic Result (GPA/CGPA)']).copy()

In [ ]:
print('Rows after cleaning:', df.shape[0])

In [ ]:
df.to_csv('../data/cleaned_survey.csv', index=False)

### 1b. Short names and encoding
- **Binary** → 0/1 (`Gender`: Male=1, `Urban`: Urban=1, `PhysAct`: Yes=1)
- **Ordinal** → numbers (`Edu`: SSC/O=1, HSC/A=2, Tertiary=3; `ParentEdu`: none=0 … tertiary=3)
- **Multi-level categorical** (`Platform`, `PeakTime`, `Withdrawal`) → dummies, but **only in the regression, not in the correlation matrix**. Rare levels are merged (TikTok, n=3, → Other; Morning, n=4, → Afternoon as "Day").
- `Timestamp`, `Year`, `Month`, `Day`, `Time` are dropped: they describe when the form was submitted, not the student.

In [ ]:
df.columns = df.columns.str.strip()
rename = {
    'Residence Area': 'Urban', 'Education Level': 'Edu',
    "Socioeconomic status (Parent's education level)": 'ParentEdu',
    'Study time (In Hours)': 'StudyH', 'Attendance rate (In Percentile)': 'Attend',
    'Social Media Platform': 'Platform', 'Time spent in social media (hours)': 'SM_hours',
    'Most time spent in a day': 'PeakTime', 'Physical activity (30 min+)': 'PhysAct',
    'Withdrawal symptoms (Side effects of not using social media)': 'Withdrawal',
    'Sleep Disturbance on Sleep Quality': 'SleepDist', 'Mood Modification Scale': 'Mood',
    'Anxiety Scale': 'Anxiety', 'Depression Scale': 'Depression',
    'Self-esteem Scale': 'SelfEsteem', 'Last Academic Result (GPA/CGPA)': 'GPA',
    'Social Media Distraction During Academic Activities': 'Distraction',
}
d = df.drop(columns=['Timestamp', 'Year', 'Month', 'Day', 'Time']).rename(columns=rename)

d['Gender']  = (d['Gender'] == 'Male').astype(int)
d['Urban']   = (d['Urban'] == 'Urban').astype(int)
d['PhysAct'] = (d['PhysAct'] == 'Yes').astype(int)
d['Edu']       = d['Edu'].map({"SSC / O' Level": 1, "HSC / A' Level": 2, "Tertiary Education": 3})
d['ParentEdu'] = d['ParentEdu'].map({'No education': 0, 'Madrasah': 1, "SSC / O' Level": 1,
                                     "HSC / A' Level": 2, 'Tertiary Education': 3})
d['Platform'] = d['Platform'].replace({'TikTok': 'Other'})
d['PeakTime'] = d['PeakTime'].replace({'Morning': 'Day', 'Afternoon': 'Day'})

print(d.isna().sum().sum(), "missing values")
for c in ['Platform', 'PeakTime', 'Withdrawal']:
    print(d[c].value_counts().to_dict())
d.describe().T

|Variable|How did they respond?|What We Did|
|---|---|---|
|Age|**21 years old**, **dd/mm/1999**|The number was extracted by converting to age (2025 − year)|
|StudyH|**3/4h**, **2 hrs**, **30 mins**|Extracted the time, minutes /60, ranges to average|
|Attend|**85%**, **0.9**, **90**|Converted to a fraction between 0 and 1|
|SM_hours|**less than 2**, **3-4 hours**, **20%**|Same as for StudyH|
|GPA|**4.0 and 5.0 scales**, **letters (A, B+, etc.)**|Everything on a 4.0 scale, Rows without a recognizable GPA have been removed|